# Zajęcia 10 — dostarczanie i strategie wdrożeń

Na zajęciach 9 powstał obraz. Teraz go wdrażamy — w sposób, który da się cofnąć.

Notebook ma dwie warstwy:

- **warstwa wykonawcza** — `compose.yaml`, router `nginx`, wdrożenie po sumie kontrolnej artefaktu, wycofanie; komórki oznaczone **[DOCKER]** wymagają działającego demona i przy jego braku wypisują komendy zamiast zgłaszać błąd;
- **warstwa decyzyjna** — generator ruchu, analiza metryk i reguła promocji albo wycofania; działa bez Dockera, bo to jest część, którą piszesz sam.

Workflow dostarczania przygotowujemy tutaj, ale uruchamiamy na GitHubie — ta część jest opisana jako „poza notebookiem".

Komórki opisane jako **zadanie** wypełniasz samodzielnie; komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE = REPO_ROOT / "artifacts" / "zajecia_10"
REGISTRY = BASE / "registry"
DEPLOY = BASE / "wdrozenie"
REPORTS = BASE / "reports"
for folder in (REGISTRY, DEPLOY / "router", DEPLOY / "env", DEPLOY / "ci", DEPLOY / ".github" / "workflows", REPORTS):
    folder.mkdir(parents=True, exist_ok=True)

IMAGE_TAG = "iris-service:1.0.0"
SEED = 42


def write_file(path: Path, content: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print(f"Zapisano: {path.relative_to(BASE)}")
    return path


def sha256_of(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def docker_status() -> tuple:
    if shutil.which("docker") is None:
        return False, "brak polecenia docker w tym srodowisku"
    probe = subprocess.run(["docker", "info"], capture_output=True, text=True)
    if probe.returncode != 0:
        return False, "demon Dockera nie odpowiada (uruchom Docker Desktop)"
    return True, "gotowy"


DOCKER_OK, DOCKER_REASON = docker_status()
print(f"Katalog pracy: {BASE.relative_to(REPO_ROOT)}")
print(f"Docker: {DOCKER_REASON}")

%load_ext skip_kernel_extension

## 2. Rejestr artefaktów

Wdrażamy **artefakt identyfikowany sumą kontrolną**, nie nazwą wersji. Nazwa `v2` może jutro oznaczać coś innego; suma kontrolna nie może.

To dokładnie ten sam mechanizm, co digest obrazu w rejestrze kontenerów. Tutaj liczymy go dla pliku modelu, bo model jest podłączany jako wolumin.

Przygotowujemy trzy modele:

| Nazwa | Czym jest | Po co |
|---|---|---|
| `champion` | model obecnie obsługujący ruch | punkt odniesienia |
| `challenger_good` | kandydat lepszy albo równie dobry | ścieżka promocji |
| `challenger_bad` | kandydat wyraźnie gorszy | sprawdzenie, czy bramka działa |

Plik `.joblib` zawiera **wyłącznie estymator** — dokładnie to, czego oczekuje obraz z zajęć 9. Lista kolumn trafia do pliku opisowego obok, bo jest metadaną wdrożenia, a nie częścią modelu.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

FEATURES = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]

frame = load_iris(as_frame=True).frame
x_train, x_holdout, y_train, y_holdout = train_test_split(
    frame[FEATURES], frame["target"], test_size=0.4, random_state=SEED, stratify=frame["target"]
)


def register(name: str, estimator, columns: list) -> dict:
    """Trenuje model, zapisuje go i rejestruje razem z suma kontrolna."""
    estimator.fit(x_train[columns], y_train)
    quality = float(f1_score(y_holdout, estimator.predict(x_holdout[columns]), average="macro"))
    path = REGISTRY / f"{name}.joblib"
    joblib.dump(estimator, path)
    entry = {
        "name": name,
        "file": path.name,
        "sha256": sha256_of(path),
        "f1_macro": round(quality, 4),
        "columns": columns,
    }
    (REGISTRY / f"{name}.json").write_text(json.dumps(entry, indent=2), encoding="utf-8")
    return entry


catalog = {
    entry["name"]: entry
    for entry in [
        register("champion", LogisticRegression(max_iter=500, random_state=SEED), FEATURES),
        register(
            "challenger_good",
            RandomForestClassifier(n_estimators=200, random_state=SEED),
            FEATURES,
        ),
        register(
            "challenger_bad",
            LogisticRegression(max_iter=500, random_state=SEED),
            ["sepal width (cm)"],
        ),
    ]
}

display(
    pd.DataFrame(catalog.values())[["name", "f1_macro", "sha256"]].assign(
        sha256=lambda d: d["sha256"].str[:16] + "..."
    )
)

## 3. Manifest wdrożenia

Stan docelowy zapisujemy deklaratywnie. Plik odpowiada na pytanie „jak ma być", a nie „co zrobić".

**Zadanie:** napisz funkcje `write_manifest(environment, champion, challenger, weight)` oraz `reconcile(manifest_path)`:

- manifest ma zawierać środowisko, sumę kontrolną artefaktu champion, opcjonalnego challengera i podział ruchu;
- `reconcile` ma **odmówić działania**, gdy suma kontrolna w manifeście nie zgadza się z plikiem w rejestrze;
- suma udziałów ruchu musi wynosić 100.

In [ ]:
# Zadanie: napisz write_manifest() i reconcile().


In [ ]:
%%skip True
MANIFEST_PATH = REPORTS / "deployment.yaml"


def write_manifest(environment: str, champion: str, challenger: str = None, weight: int = 0) -> Path:
    if not 0 <= weight <= 100:
        raise ValueError("Udział ruchu musi mieścić się w przedziale 0-100.")
    if challenger is None and weight != 0:
        raise ValueError("Nie można skierować ruchu do nieistniejącego kandydata.")

    manifest = {
        "environment": environment,
        "image": IMAGE_TAG,
        "model": {
            "champion": {"name": champion, "sha256": catalog[champion]["sha256"]},
        },
        "traffic": {"champion": 100 - weight, "challenger": weight},
    }
    if challenger is not None:
        manifest["model"]["challenger"] = {
            "name": challenger,
            "sha256": catalog[challenger]["sha256"],
        }

    MANIFEST_PATH.write_text(yaml.safe_dump(manifest, sort_keys=False), encoding="utf-8")
    return MANIFEST_PATH


def reconcile(manifest_path: Path) -> dict:
    """Sprawdza zgodnosc stanu opisanego ze stanem rzeczywistym w rejestrze."""
    manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
    problems = []

    if sum(manifest["traffic"].values()) != 100:
        problems.append("Udziały ruchu nie sumują się do 100.")

    for role, spec in manifest["model"].items():
        artifact = REGISTRY / f"{spec['name']}.joblib"
        if not artifact.exists():
            problems.append(f"{role}: brak artefaktu {artifact.name} w rejestrze.")
            continue
        actual = sha256_of(artifact)
        if actual != spec["sha256"]:
            problems.append(
                f"{role}: suma kontrolna niezgodna (manifest {spec['sha256'][:12]}, "
                f"rejestr {actual[:12]})"
            )

    if problems:
        for problem in problems:
            print(f"  ODMOWA: {problem}")
        raise RuntimeError("Uzgadnianie przerwane — stan opisany nie odpowiada rzeczywistemu.")

    print(f"Uzgodniono {manifest['environment']}: {manifest['traffic']}")
    return manifest


write_manifest("production", champion="champion")
current = reconcile(MANIFEST_PATH)
print()
print(MANIFEST_PATH.read_text(encoding="utf-8"))

## 4. Wykrycie podmiany artefaktu

Manifest, którego nikt nie weryfikuje, jest tylko komentarzem. Sprawdzamy, czy uzgadnianie zauważy, że plik modelu zmienił się bez zmiany manifestu.

**Do opisania:** jakie realne zdarzenie odpowiada tej sytuacji w projekcie?

In [ ]:
champion_path = REGISTRY / "champion.joblib"
backup = champion_path.read_bytes()

shutil.copyfile(REGISTRY / "challenger_bad.joblib", champion_path)
try:
    reconcile(REPORTS / "deployment.yaml")
except RuntimeError as error:
    print(f"\n{error}")
finally:
    champion_path.write_bytes(backup)

print("\nPo przywróceniu pliku:")
reconcile(REPORTS / "deployment.yaml")

## 5. Środowiska w `docker compose`

Staging i produkcja to dwa zestawy tych samych usług, różniące się wyłącznie konfiguracją. Router `nginx` rozdziela ruch między champion i challengera według wag.

**Zadanie:** zapisz `compose.yaml`, `router/nginx.conf` oraz pliki `env/staging.env` i `env/production.env`. Wymagania:

- obraz jest jeden i ten sam dla obu wersji usługi — różni je wyłącznie podłączony model,
- katalog z modelami montujemy tylko do odczytu,
- port publiczny i nazwa projektu pochodzą z pliku środowiska,
- wagi routera dają się zmienić bez restartu usług.

Dwie pułapki, na które warto uważać:

- `nginx` odrzuca `weight=0` — wyłączenie wersji z ruchu zapisuje się znacznikiem `down`, nie zerową wagą;
- ścieżki w systemie Windows zawierają dwukropek po literze dysku, więc wolumin opisujemy **składnią rozszerzoną**, a nie skróconą.

In [ ]:
# Zadanie: opisz środowisko w compose.yaml i skonfiguruj router.


In [ ]:
%%skip True
write_file(
    DEPLOY / "compose.yaml",
    """
    # Jeden obraz, dwie wersje uslugi, jeden router.
    # Uruchomienie: docker compose --env-file env/staging.env up -d
    services:
      api-champion:
        image: ${IMAGE}
        environment:
          MODEL_PATH: /app/models/${CHAMPION_FILE}
          LOG_LEVEL: ${LOG_LEVEL}
        volumes:
          - type: bind
            source: ${MODELS_DIR}
            target: /app/models
            read_only: true
        expose:
          - "8000"

      api-challenger:
        image: ${IMAGE}
        environment:
          MODEL_PATH: /app/models/${CHALLENGER_FILE}
          LOG_LEVEL: ${LOG_LEVEL}
        volumes:
          - type: bind
            source: ${MODELS_DIR}
            target: /app/models
            read_only: true
        expose:
          - "8000"

      router:
        image: nginx:1.27-alpine
        depends_on:
          - api-champion
          - api-challenger
        volumes:
          - type: bind
            source: ./router/nginx.conf
            target: /etc/nginx/nginx.conf
            read_only: true
        ports:
          - "${PUBLIC_PORT}:80"
    """,
)

write_file(
    DEPLOY / "router" / "nginx.conf",
    """
    # Podzial ruchu steruje sie wagami. Znacznik `down` wylacza wersje calkowicie;
    # `weight=0` jest bledem skladni i nginx sie nie uruchomi.
    # Canary 10%: usun `down` i ustaw weight=10 przy champion weight=90,
    # nastepnie `nginx -s reload` - bez restartu uslug i bez utraty zadan.
    events {}

    http {
      upstream backend {
        server api-champion:8000 weight=100;
        server api-challenger:8000 weight=1 down;
      }

      server {
        listen 80;

        location / {
          proxy_pass http://backend;
          proxy_set_header Host $host;
          proxy_connect_timeout 2s;
          proxy_read_timeout 5s;
        }
      }
    }
    """,
)

models_dir = str(REGISTRY).replace("\\", "/")
for environment, port, level in (("staging", 8081, "debug"), ("production", 8080, "info")):
    write_file(
        DEPLOY / "env" / f"{environment}.env",
        f"""
        COMPOSE_PROJECT_NAME=iris-{environment}
        ENVIRONMENT={environment}
        IMAGE={IMAGE_TAG}
        MODELS_DIR={models_dir}
        CHAMPION_FILE=champion.joblib
        CHALLENGER_FILE=challenger_good.joblib
        PUBLIC_PORT={port}
        LOG_LEVEL={level}
        """,
    )

## 6. Uruchomienie środowiska staging **[DOCKER]**

Uruchamiamy opisany stan i sprawdzamy, czy usługa odpowiada przez router. Wymagany jest obraz `iris-service:1.0.0` z zajęć 9.

**Do opisania:** ile trwa doprowadzenie środowiska do stanu gotowego i co dokładnie sprawdzamy, zanim uznamy je za działające?

In [ ]:
COMPOSE_STAGING = [
    "docker", "compose", "--env-file", "env/staging.env", "-f", "compose.yaml",
]
STAGING_URL = "http://127.0.0.1:8081"


def image_present(tag: str) -> bool:
    probe = subprocess.run(["docker", "image", "inspect", tag], capture_output=True, text=True)
    return probe.returncode == 0


def compose(*arguments) -> subprocess.CompletedProcess:
    return subprocess.run(
        COMPOSE_STAGING + list(arguments), cwd=DEPLOY, capture_output=True, text=True
    )


if not DOCKER_OK:
    print(f"[DOCKER] pominięte — {DOCKER_REASON}")
    print("Uruchom samodzielnie w katalogu wdrozenie/:")
    print("  " + " ".join(COMPOSE_STAGING) + " up -d")
    print(f"  curl {STAGING_URL}/health")
    print("  " + " ".join(COMPOSE_STAGING) + " down")
elif not image_present(IMAGE_TAG):
    print(f"[DOCKER] brak obrazu {IMAGE_TAG} — zbuduj go w notebooku zajęć 9.")
else:
    import httpx

    started = time.perf_counter()
    launched = compose("up", "-d")
    if launched.returncode != 0:
        print(launched.stderr[-1500:])
        raise RuntimeError("docker compose up zakończyło się błędem.")

    try:
        for _ in range(60):
            try:
                if httpx.get(f"{STAGING_URL}/health", timeout=1.0).status_code == 200:
                    break
            except Exception:
                time.sleep(0.5)
        else:
            print(compose("logs", "--tail", "20").stdout[-1500:])
            raise RuntimeError("Środowisko nie odpowiedziało w wyznaczonym czasie.")

        print(f"Gotowe w {time.perf_counter() - started:.1f} s")
        print(f"/health -> {httpx.get(STAGING_URL + '/health').json()}")
        print(f"/ready  -> {httpx.get(STAGING_URL + '/ready').json()}")

        sample = {
            "records": [
                {
                    "sepal_length_cm": 5.1,
                    "sepal_width_cm": 3.5,
                    "petal_length_cm": 1.4,
                    "petal_width_cm": 0.2,
                }
            ]
        }
        answer = httpx.post(f"{STAGING_URL}/v1/predict", json=sample, timeout=10.0).json()
        print(f"/v1/predict -> {answer}")
    finally:
        compose("down")
        print("Środowisko zatrzymane.")

## 7. Router i generator ruchu

Dalsza część dotyczy **decyzji**, nie infrastruktury, więc pracujemy w pamięci. Router kieruje żądanie do jednej z wersji zgodnie z wagą i zapisuje wynik: wersję, poprawność predykcji, opóźnienie i ewentualny błąd.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
class Router:
    """Kieruje ruch miedzy champion a challenger zgodnie z waga."""

    def __init__(self, champion_name: str, challenger_name: str = None, weight: int = 0, seed: int = SEED):
        self.versions = {"champion": self._load(champion_name)}
        if challenger_name is not None:
            self.versions["challenger"] = self._load(challenger_name)
        self.weight = weight
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def _load(name: str) -> dict:
        entry = json.loads((REGISTRY / f"{name}.json").read_text(encoding="utf-8"))
        return {
            "name": name,
            "estimator": joblib.load(REGISTRY / entry["file"]),
            "columns": entry["columns"],
        }

    def set_weight(self, weight: int) -> None:
        if weight > 0 and "challenger" not in self.versions:
            raise ValueError("Brak kandydata, do którego można skierować ruch.")
        self.weight = weight

    def handle(self, record: pd.Series, truth: int) -> dict:
        target = "challenger" if self.rng.integers(0, 100) < self.weight else "champion"
        version = self.versions[target]
        started = time.perf_counter()
        try:
            prediction = int(
                version["estimator"].predict(record[version["columns"]].to_frame().T)[0]
            )
            error = None
        except Exception as exc:
            prediction, error = None, type(exc).__name__
        latency_ms = (time.perf_counter() - started) * 1000
        return {
            "wersja": target,
            "model": version["name"],
            "predykcja": prediction,
            "prawda": truth,
            "trafiona": prediction == truth,
            "blad": error,
            "opoznienie_ms": round(latency_ms, 3),
        }


def generate_traffic(router: Router, requests: int, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(x_holdout), size=requests)
    rows = [
        router.handle(x_holdout.iloc[index], int(y_holdout.iloc[index])) for index in indices
    ]
    return pd.DataFrame(rows)


probe = generate_traffic(Router("champion", "challenger_good", weight=20), requests=200)
display(probe.groupby("wersja").agg(zadania=("trafiona", "size"), trafnosc=("trafiona", "mean")))

## 8. Analiza canary

Wdrożenie canary bez automatycznej analizy jest tylko wolniejszym wdrożeniem. Wartość pojawia się dopiero wtedy, gdy zdefiniowano kryteria zatrzymania.

**Zadanie:** napisz `analyze_canary(traffic, min_observations, max_quality_drop, max_error_rate)` zwracającą decyzję **`promuj`**, **`czekaj`** albo **`wycofaj`** wraz z uzasadnieniem. Wymagania:

- przy liczbie obserwacji kandydata poniżej progu decyzja brzmi `czekaj` — bez wyjątków;
- spadek jakości większy niż `max_quality_drop` względem champion oznacza `wycofaj`;
- udział błędów powyżej `max_error_rate` oznacza `wycofaj`;
- w pozostałych przypadkach `promuj`.

In [ ]:
# Zadanie: napisz analyze_canary().


In [ ]:
%%skip True
def analyze_canary(
    traffic: pd.DataFrame,
    min_observations: int = 100,
    max_quality_drop: float = 0.02,
    max_error_rate: float = 0.01,
) -> dict:
    challenger = traffic[traffic["wersja"] == "challenger"]
    champion = traffic[traffic["wersja"] == "champion"]

    metrics = {
        "obserwacje_kandydata": int(len(challenger)),
        "trafnosc_champion": round(float(champion["trafiona"].mean()), 4) if len(champion) else None,
        "trafnosc_kandydata": round(float(challenger["trafiona"].mean()), 4) if len(challenger) else None,
        "udzial_bledow": round(float(challenger["blad"].notna().mean()), 4) if len(challenger) else 0.0,
        "opoznienie_p95_ms": round(float(challenger["opoznienie_ms"].quantile(0.95)), 3)
        if len(challenger)
        else None,
    }

    if metrics["obserwacje_kandydata"] < min_observations:
        return {
            "decyzja": "czekaj",
            "powod": f"za mało obserwacji ({metrics['obserwacje_kandydata']} < {min_observations})",
            **metrics,
        }

    if metrics["udzial_bledow"] > max_error_rate:
        return {
            "decyzja": "wycofaj",
            "powod": f"udział błędów {metrics['udzial_bledow']:.3f} > {max_error_rate}",
            **metrics,
        }

    drop = metrics["trafnosc_champion"] - metrics["trafnosc_kandydata"]
    if drop > max_quality_drop:
        return {
            "decyzja": "wycofaj",
            "powod": f"spadek jakości {drop:.3f} > {max_quality_drop}",
            **metrics,
        }

    return {"decyzja": "promuj", "powod": f"różnica jakości {drop:+.3f} w granicach tolerancji", **metrics}


for label, name in (("kandydat dobry", "challenger_good"), ("kandydat gorszy", "challenger_bad")):
    sample = generate_traffic(Router("champion", name, weight=50), requests=600)
    verdict = analyze_canary(sample)
    print(f"{label:16s} -> {verdict['decyzja']:8s} ({verdict['powod']})")

## 9. Wdrożenie canary krok po kroku

Canary to sekwencja etapów: zwiększamy udział ruchu tylko wtedy, gdy analiza na poprzednim etapie wypadła pomyślnie.

**Zadanie:** napisz `run_canary(challenger, steps, requests_per_step)`, która:

- na każdym etapie ustawia wagę, generuje ruch i wywołuje analizę,
- przerywa przy decyzji `wycofaj` i przywraca wagę 0,
- przy decyzji `czekaj` dobiera ruch, zamiast zwiększać udział,
- zapisuje pełny ślad decyzji do `reports/canary_analysis.json`.

In [ ]:
# Zadanie: napisz run_canary().


In [ ]:
%%skip True
def run_canary(
    challenger: str,
    steps: tuple = (5, 25, 50, 100),
    requests_per_step: int = 800,
    min_observations: int = 100,
) -> dict:
    router = Router("champion", challenger, weight=0)
    trace = []

    for index, weight in enumerate(steps):
        router.set_weight(weight)
        traffic = generate_traffic(router, requests_per_step, seed=SEED + index)
        verdict = analyze_canary(traffic, min_observations=min_observations)
        trace.append({"etap": index + 1, "udzial_ruchu": weight, **verdict})
        print(
            f"etap {index + 1}: ruch {weight:3d}% -> {verdict['decyzja']:8s} "
            f"({verdict['powod']})"
        )

        if verdict["decyzja"] == "wycofaj":
            router.set_weight(0)
            outcome = "wycofany"
            break
        if verdict["decyzja"] == "czekaj":
            traffic = generate_traffic(router, requests_per_step * 3, seed=SEED + 100 + index)
            verdict = analyze_canary(traffic, min_observations=min_observations)
            trace.append({"etap": index + 1, "udzial_ruchu": weight, "dobrany_ruch": True, **verdict})
            print(f"        po dobraniu ruchu -> {verdict['decyzja']}")
            if verdict["decyzja"] == "wycofaj":
                router.set_weight(0)
                outcome = "wycofany"
                break
    else:
        outcome = "promowany"

    result = {
        "kandydat": challenger,
        "wynik": outcome,
        "koncowy_udzial_ruchu": router.weight,
        "slad": trace,
    }
    (REPORTS / "canary_analysis.json").write_text(
        json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"\nWynik: {outcome} (ruch kandydata {router.weight}%)")
    return result


run_canary("challenger_good")

## 10. Punkt kontrolny 1 — gorszy kandydat

Ten sam proces, kandydat wyraźnie gorszy od obecnego modelu.

**Do opisania:** na którym etapie proces się zatrzymał, ile żądań obsłużył gorszy model i jaki byłby ten wynik przy wdrożeniu bez etapów?

In [ ]:
outcome = run_canary("challenger_bad")

exposed = sum(row["obserwacje_kandydata"] for row in outcome["slad"])
print(f"\nŻądań obsłużonych przez gorszy model: {exposed}")
print(f"Przy wdrożeniu bez etapów byłoby ich: {800 * 4}")
display(pd.DataFrame(outcome["slad"])[["etap", "udzial_ruchu", "decyzja", "powod"]])

## 11. Tryb shadow

W trybie shadow kandydat dostaje **kopię** ruchu, ale jego odpowiedzi nie trafiają do użytkownika. Odpowiada to na pytanie „czy nowa wersja zachowuje się rozsądnie" bez żadnego ryzyka.

Cena: podwójne obliczenia oraz to, że nie zmierzymy w ten sposób wpływu na zachowanie użytkownika.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
def run_shadow(challenger: str, requests: int = 1000) -> pd.DataFrame:
    primary = Router("champion")
    shadow = Router(challenger)
    rng = np.random.default_rng(SEED)
    indices = rng.integers(0, len(x_holdout), size=requests)

    rows = []
    for index in indices:
        record, truth = x_holdout.iloc[index], int(y_holdout.iloc[index])
        served = primary.handle(record, truth)
        observed = shadow.handle(record, truth)
        rows.append(
            {
                "predykcja_produkcyjna": served["predykcja"],
                "predykcja_shadow": observed["predykcja"],
                "zgodne": served["predykcja"] == observed["predykcja"],
                "trafiona_produkcyjna": served["trafiona"],
                "trafiona_shadow": observed["trafiona"],
            }
        )
    return pd.DataFrame(rows)


summary = []
for name in ("challenger_good", "challenger_bad"):
    result = run_shadow(name)
    summary.append(
        {
            "kandydat": name,
            "zgodnosc_decyzji": round(float(result["zgodne"].mean()), 4),
            "trafnosc_produkcyjna": round(float(result["trafiona_produkcyjna"].mean()), 4),
            "trafnosc_kandydata": round(float(result["trafiona_shadow"].mean()), 4),
        }
    )

display(pd.DataFrame(summary))
print("Żaden użytkownik nie zobaczył odpowiedzi kandydata.")

## 12. Punkt kontrolny 2 — złamana zgodność wsteczna

Nowa wersja usługi zmienia format odpowiedzi: zamiast `{"prediction": 0, "label": "setosa"}` zwraca `{"class": "setosa"}`. Model działa poprawnie, ale konsument przestaje rozumieć wynik.

**Zadanie:** napisz `contract_test(response)` sprawdzający kontrakt odpowiedzi i uruchom go na obu wersjach. Zastanów się, czy przełączenie wagi routera wystarczy, żeby naprawić sytuację.

In [ ]:
LABELS = ["setosa", "versicolor", "virginica"]


def respond_v1(prediction: int) -> dict:
    return {"contract_version": "v1", "prediction": prediction, "label": LABELS[prediction]}


def respond_v2(prediction: int) -> dict:
    return {"contract_version": "v2", "class": LABELS[prediction]}


# Zadanie: napisz contract_test(response) i sprawdź obie wersje.


In [ ]:
%%skip True
REQUIRED_FIELDS = {"prediction": int, "label": str}


def contract_test(response: dict) -> list:
    problems = []
    for field, expected in REQUIRED_FIELDS.items():
        if field not in response:
            problems.append(f"brak pola '{field}'")
        elif not isinstance(response[field], expected):
            problems.append(f"pole '{field}' ma typ {type(response[field]).__name__}")
    if "label" in response and response["label"] not in LABELS:
        problems.append("nieznana wartość pola 'label'")
    return problems


for name, responder in (("v1", respond_v1), ("v2", respond_v2)):
    problems = contract_test(responder(0))
    verdict = "OK" if not problems else "NARUSZENIE: " + ", ".join(problems)
    print(f"{name}: {responder(0)}")
    print(f"    {verdict}\n")

print(
    "Przełączenie wagi routera cofa ruch do wersji v1, ale nie naprawia klientów,\n"
    "którzy w międzyczasie dostali odpowiedź w formacie v2 i ją zapisali.\n"
    "Zmiana modelu i zmiana kontraktu to dwa osobne wdrożenia."
)

## 13. Punkt kontrolny 3 — wycofanie

Wycofanie polega na ponownym wdrożeniu **poprzedniego artefaktu**, identyfikowanego sumą kontrolną. Nic nie budujemy — poprzednia wersja jest nadal w rejestrze.

Mierzymy czas od decyzji do potwierdzenia poprawnego działania.

**Do opisania:** co w twoim projekcie zajęłoby tu najwięcej czasu?

In [ ]:
def deploy(champion: str, challenger: str = None, weight: int = 0) -> dict:
    write_manifest("production", champion=champion, challenger=challenger, weight=weight)
    return reconcile(REPORTS / "deployment.yaml")


def verify(champion: str, requests: int = 400, min_accuracy: float = 0.90) -> bool:
    traffic = generate_traffic(Router(champion), requests)
    accuracy = float(traffic["trafiona"].mean())
    print(f"    kontrola po wdrożeniu: trafność {accuracy:.4f}")
    return accuracy >= min_accuracy


print("Stan wyjściowy — omyłkowe wdrożenie gorszego modelu jako champion:")
deploy(champion="challenger_bad")
bad_ok = verify("challenger_bad")
print(f"    stan poprawny: {bad_ok}\n")

print("Wycofanie:")
started = time.perf_counter()
deploy(champion="champion")
restored = verify("champion")
elapsed = time.perf_counter() - started

timing = {
    "czas_wycofania_s": round(elapsed, 3),
    "stan_poprawny_po_wycofaniu": bool(restored),
    "metoda": "ponowne wdrożenie poprzedniej sumy kontrolnej z rejestru",
}
(REPORTS / "rollback_timing.json").write_text(
    json.dumps(timing, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"\nCzas wycofania: {elapsed:.3f} s")

## 14. Workflow dostarczania

Ostatni element to opis procesu w GitHub Actions. Trzy rzeczy są tu istotne:

- wdrażamy **po digeście**, nie po znaczniku — digest ustalamy raz i przekazujemy między zadaniami;
- staging wdraża się automatycznie, produkcja czeka na zatwierdzenie (`environment: production`);
- polecenia wykonuje **self-hosted runner** na maszynie docelowej, więc GitHub nie potrzebuje do niej dostępu.

**Zadanie:** uzupełnij poniższy szkielet o ustalenie digestu, o zadanie `deploy-production` z bramką zatwierdzenia oraz o krok weryfikacji po wdrożeniu.

In [ ]:
# Zadanie: uzupełnij workflow dostarczania.
CD_PATH = DEPLOY / ".github" / "workflows" / "cd.yml"
write_file(
    CD_PATH,
    """
    name: cd

    on:
      workflow_run:
        workflows: [ci]
        branches: [main]
        types: [completed]

    jobs:
      deploy-staging:
        if: ${{ github.event.workflow_run.conclusion == 'success' }}
        runs-on: [self-hosted, demo]
        environment: staging
        steps:
          - uses: actions/checkout@v4
          - name: Wdrozenie na staging
            run: docker compose --env-file env/staging.env up -d
    """,
)
print(CD_PATH.read_text(encoding="utf-8"))

In [ ]:
%%skip True
write_file(
    CD_PATH,
    """
    name: cd

    on:
      workflow_run:
        workflows: [ci]
        branches: [main]
        types: [completed]

    permissions:
      contents: read
      packages: read

    env:
      IMAGE: ghcr.io/${{ github.repository }}/iris-service

    jobs:
      # Digest ustalamy raz i przekazujemy dalej - staging i produkcja
      # dostaja dokladnie ten sam artefakt, nie tylko ten sam znacznik.
      resolve-image:
        if: ${{ github.event.workflow_run.conclusion == 'success' }}
        runs-on: ubuntu-latest
        outputs:
          reference: ${{ steps.resolve.outputs.reference }}
        steps:
          - name: Logowanie do rejestru
            uses: docker/login-action@v3
            with:
              registry: ghcr.io
              username: ${{ github.actor }}
              password: ${{ secrets.GITHUB_TOKEN }}
          - name: Ustalenie digestu
            id: resolve
            run: |
              SHA=${{ github.event.workflow_run.head_sha }}
              DIGEST=$(docker buildx imagetools inspect "$IMAGE:$SHA" --format '{{json .Manifest.Digest}}' | tr -d '"')
              echo "reference=$IMAGE@$DIGEST" >> "$GITHUB_OUTPUT"
              echo "Wdrazany artefakt: $IMAGE@$DIGEST" >> "$GITHUB_STEP_SUMMARY"

      deploy-staging:
        needs: resolve-image
        runs-on: [self-hosted, demo]
        environment: staging
        steps:
          - uses: actions/checkout@v4
          - name: Wdrozenie na staging
            run: docker compose --env-file env/staging.env up -d
            env:
              IMAGE: ${{ needs.resolve-image.outputs.reference }}
          - name: Kontrola po wdrozeniu
            run: ./ci/verify_deployment.sh http://127.0.0.1:8081

      # Bramka zatwierdzenia: srodowisko `production` ma regule
      # `required reviewers`, wiec to zadanie czeka na decyzje czlowieka.
      deploy-production:
        needs: [resolve-image, deploy-staging]
        runs-on: [self-hosted, demo]
        environment:
          name: production
          url: http://127.0.0.1:8080/health
        steps:
          - uses: actions/checkout@v4
          - name: Zapis obecnego artefaktu na potrzeby wycofania
            run: docker compose --env-file env/production.env config --images | head -1 > .previous-image
          - name: Wdrozenie na produkcje
            run: docker compose --env-file env/production.env up -d
            env:
              IMAGE: ${{ needs.resolve-image.outputs.reference }}
          - name: Kontrola po wdrozeniu
            run: ./ci/verify_deployment.sh http://127.0.0.1:8080
          - name: Wycofanie przy niepowodzeniu kontroli
            if: failure()
            run: |
              PREVIOUS=$(cat .previous-image)
              IMAGE="$PREVIOUS" docker compose --env-file env/production.env up -d
              ./ci/verify_deployment.sh http://127.0.0.1:8080
              echo "Wycofano do $PREVIOUS" >> "$GITHUB_STEP_SUMMARY"
    """,
)

write_file(
    DEPLOY / "ci" / "verify_deployment.sh",
    """
    #!/usr/bin/env bash
    # Kontrola po wdrozeniu: usluga zyje, model jest zaladowany, predykcja dziala.
    set -euo pipefail
    BASE="$1"

    for _ in $(seq 1 30); do
      if curl -fsS "$BASE/health" >/dev/null 2>&1; then break; fi
      sleep 2
    done

    curl -fsS "$BASE/ready" >/dev/null
    curl -fsS -X POST "$BASE/v1/predict" \\
      -H 'Content-Type: application/json' \\
      -d '{"records":[{"sepal_length_cm":5.1,"sepal_width_cm":3.5,"petal_length_cm":1.4,"petal_width_cm":0.2}]}' \\
      | grep -q 'predictions'

    echo "Kontrola po wdrozeniu zakonczona powodzeniem."
    """,
)

print()
print(CD_PATH.read_text(encoding="utf-8")[:900])

## 15. Procedura wycofania i model zagrożeń

Dwa dokumenty, które muszą powstać **przed** incydentem, a nie w jego trakcie.

**Zadanie:** zapisz `reports/rollback_runbook.md` oraz `reports/threat_model.md`. Runbook ma zawierać warunek uruchomienia, role, kroki, sposób potwierdzenia i postępowanie z danymi z okresu awarii. Model zagrożeń — kategorie ryzyka wraz z decyzją dla każdego z nich.

In [ ]:
# Zadanie: napisz procedurę wycofania i model zagrożeń.


In [ ]:
%%skip True
measured = json.loads((REPORTS / "rollback_timing.json").read_text(encoding="utf-8"))

write_file(
    REPORTS / "rollback_runbook.md",
    f"""
    # Procedura wycofania — usługa iris-service

    ## Warunek uruchomienia

    Wycofujemy, gdy zachodzi co najmniej jeden warunek:

    - udział odpowiedzi z błędem przekracza 1% w oknie 5 minut,
    - trafność na ruchu z etykietami spada o więcej niż 2 punkty względem champion,
    - opóźnienie p95 przekracza 300 ms,
    - kontrola po wdrożeniu (`verify_deployment.sh`) kończy się niepowodzeniem.

    ## Role

    - **decyzja:** dyżurny inżynier, bez konsultacji, na podstawie powyższych progów;
    - **wykonanie:** workflow `cd` uruchomiony ponownie z poprzednim digestem;
    - **powiadomienie:** właściciel produktu, po przywróceniu działania.

    ## Kroki

    1. Odczytaj poprzedni digest z podsumowania ostatniego udanego przebiegu.
    2. Uruchom wdrożenie z tym digestem:
       `IMAGE=<digest> docker compose --env-file env/production.env up -d`.
    3. Uruchom `ci/verify_deployment.sh http://127.0.0.1:8080`.
    4. Potwierdź, że metryki wróciły do poziomu sprzed wdrożenia.

    ## Potwierdzenie

    Wycofanie uznajemy za skuteczne, gdy kontrola po wdrożeniu przechodzi,
    a trafność na próbce kontrolnej wraca powyżej 0.90.
    Zmierzony czas wycofania w warunkach ćwiczenia: **{measured['czas_wycofania_s']} s**.

    ## Dane z okresu awarii

    Predykcje wykonane w oknie awarii oznaczamy znacznikiem `suspect=true`
    i **wykluczamy ze zbioru treningowego** do czasu przeglądu.
    Bez tego kroku błędna wersja modelu wraca do systemu przez dane.

    ## Czego ta procedura nie obejmuje

    Zmiany łamiącej kontrakt odpowiedzi — jej wycofanie wymaga działania
    po stronie konsumentów i nie da się go wykonać samym przełączeniem wersji.
    """,
)

write_file(
    REPORTS / "threat_model.md",
    """
    # Model zagrożeń — usługa predykcyjna

    | Zagrożenie | Na czym polega | Decyzja |
    |---|---|---|
    | Nadużycie endpointu | masowe odpytywanie w celu darmowych predykcji | ograniczamy: uwierzytelnianie + limit 100 żądań/min |
    | Kradzież modelu | odtworzenie modelu z odpowiedzi | ograniczamy częściowo: zwracamy klasę, nie rozkład prawdopodobieństw |
    | Wnioskowanie o danych treningowych | ustalenie, czy rekord był w zbiorze | akceptujemy: dane publiczne, brak informacji wrażliwych |
    | Dane trujące | wpływ na model przez dane zwrotne | ograniczamy: dane z okresu awarii oznaczane i wykluczane |
    | Wejście adwersarialne | spreparowane dane wymuszające błąd | ograniczamy: walidacja zakresów w modelu żądania |
    | Podmiana artefaktu | wdrożenie modelu z niezaufanego źródła | ograniczamy: wdrożenie po sumie kontrolnej, atestacja pochodzenia obrazu |

    ## Ryzyka zaakceptowane świadomie

    - brak szyfrowania ruchu wewnątrz sieci compose — środowisko demonstracyjne;
    - brak rotacji poświadczeń rejestru — token przebiegu wygasa samoczynnie.

    ## Ryzyko o najwyższym priorytecie

    Podmiana artefaktu. Jest jedynym zagrożeniem, które pozwala napastnikowi
    wykonać własny kod w środowisku produkcyjnym, a nie tylko wpłynąć na wynik.
    """,
)

## 16. Poza notebookiem — konfiguracja na GitHubie

### Krok 1 — środowiska i bramka zatwierdzenia

Settings → Environments:

1. utwórz `staging` — bez reguł;
2. utwórz `production` — włącz **Required reviewers** i wskaż osobę zatwierdzającą;
3. opcjonalnie ustaw **Wait timer**, jeżeli chcesz wymusić odstęp między wdrożeniami.

Po tej zmianie zadanie `deploy-production` zatrzyma się i będzie czekać. To jest różnica między ciągłym **dostarczaniem** a ciągłym **wdrażaniem**.

### Krok 2 — runner na maszynie docelowej

Settings → Actions → Runners → New self-hosted runner. Postępuj zgodnie z instrukcją dla swojego systemu i nadaj runnerowi etykietę `demo`, zgodną z `runs-on: [self-hosted, demo]`.

Zwróć uwagę na kierunek połączenia: to runner łączy się z GitHubem. Maszyna docelowa nie musi być dostępna z internetu i nie przekazujemy nikomu poświadczeń administracyjnych.

### Krok 3 — przebieg pełnej ścieżki

1. Wprowadź zmianę na gałęzi i otwórz zgłoszenie zmiany — ma uruchomić się tylko `ci`.
2. Scal do `main` — po zakończeniu `ci` uruchamia się `cd`.
3. Obejrzyj `deploy-staging`, potem zatwierdź `deploy-production`.
4. Zapisz digest z podsumowania przebiegu — będzie potrzebny do wycofania.

### Krok 4 — próba wycofania

Wdróż celowo wadliwą wersję, doprowadź do niepowodzenia kontroli po wdrożeniu i **zmierz czas** powrotu do poprzedniego digestu. Zapisz tę liczbę w sprawozdaniu.

## 17. Podsumowanie

Projekt ma opisany deklaratywnie stan docelowy, uzgadnianie odmawiające działania przy niezgodnej sumie kontrolnej, wdrożenie etapowe z automatyczną decyzją, tryb shadow, przetestowaną procedurę wycofania i model zagrożeń.

Najważniejsza obserwacja z punktu kontrolnego 1: wdrożenie etapowe nie zapobiega złym modelom. Ono **ogranicza liczbę osób, które zobaczą skutki** — i to jest cała jego wartość.

Czego nadal brakuje:

- decyzje opierają się na etykietach, które w produkcji przychodzą z opóźnieniem albo wcale (zajęcia 11 i 12);
- nikt nie obserwuje usługi między wdrożeniami (zajęcia 11);
- nie ma sygnału, który uruchomiłby ponowny trening (zajęcia 12).

### Zadanie do własnego projektu

1. Deklaratywny opis stanu docelowego w repozytorium.
2. Wdrożenie **po digeście**, ten sam artefakt na staging i produkcji.
3. Workflow dostarczania: staging automatycznie, produkcja za bramką zatwierdzenia.
4. Jedna strategia bezpiecznej zmiany wersji: canary, shadow albo blue-green.
5. Kryteria automatycznej promocji i wycofania, z minimalną liczbą obserwacji.
6. Procedura wycofania **wykonana co najmniej raz**, ze zmierzonym czasem.
7. Model zagrożeń z decyzją dla każdego ryzyka.
8. Lista zmian, których **nie** da się wycofać samym przełączeniem wersji modelu.

**Kryterium ukończenia:** wprowadzenie gorszej wersji modelu kończy się automatycznym wycofaniem bez udziału człowieka, a ślad tej decyzji jest zapisany.